<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/MLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


# ---------------------------------------------------------
# 1. Load the training data
# ---------------------------------------------------------
with open("training_data.txt", "r", encoding="utf-8") as file:
    sentences = [
        line.strip().lower()
        for line in file
        if line.strip()
    ]


# ---------------------------------------------------------
# 2. Build the vocabulary
# ---------------------------------------------------------
special_tokens = ["[PAD]", "[MASK]", "[UNK]"]

all_words = []
for sentence in sentences:
    all_words.extend(sentence.split())

word_counts = Counter(all_words)

vocabulary = special_tokens + sorted(word_counts.keys())

word_to_id = {
    word: index
    for index, word in enumerate(vocabulary)
}

id_to_word = {
    index: word
    for word, index in word_to_id.items()
}

PAD_ID = word_to_id["[PAD]"]
MASK_ID = word_to_id["[MASK]"]
UNK_ID = word_to_id["[UNK]"]

VOCAB_SIZE = len(vocabulary)

print(f"Vocabulary size: {VOCAB_SIZE}")


# ---------------------------------------------------------
# 3. Convert sentences into token IDs
# ---------------------------------------------------------
def tokenize(sentence: str) -> list[int]:
    return [
        word_to_id.get(word, UNK_ID)
        for word in sentence.lower().split()
    ]


tokenized_sentences = [
    tokenize(sentence)
    for sentence in sentences
]

MAX_LENGTH = max(len(sentence) for sentence in tokenized_sentences)


def pad_sequence(sequence: list[int]) -> list[int]:
    padding_length = MAX_LENGTH - len(sequence)
    return sequence + [PAD_ID] * padding_length


# ---------------------------------------------------------
# 4. Create masked training examples
# ---------------------------------------------------------
class MaskedLanguageDataset(Dataset):
    def __init__(self, tokenized_data):
        self.data = tokenized_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        original_tokens = self.data[index].copy()

        valid_positions = [
            position
            for position, token_id in enumerate(original_tokens)
            if token_id != PAD_ID
        ]

        mask_position = random.choice(valid_positions)

        target_word_id = original_tokens[mask_position]

        masked_tokens = original_tokens.copy()
        masked_tokens[mask_position] = MASK_ID

        masked_tokens = pad_sequence(masked_tokens)

        return {
            "input_ids": torch.tensor(
                masked_tokens,
                dtype=torch.long
            ),
            "mask_position": torch.tensor(
                mask_position,
                dtype=torch.long
            ),
            "target": torch.tensor(
                target_word_id,
                dtype=torch.long
            )
        }


dataset = MaskedLanguageDataset(tokenized_sentences)

data_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)


# ---------------------------------------------------------
# 5. Define the MLM model
# ---------------------------------------------------------
class SimpleMaskedLanguageModel(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        embedding_size=64,
        hidden_size=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_size,
            padding_idx=PAD_ID
        )

        self.encoder = nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.output_layer = nn.Linear(
            hidden_size * 2,
            vocabulary_size
        )

    def forward(self, input_ids, mask_positions):
        embeddings = self.embedding(input_ids)

        encoded_output, _ = self.encoder(embeddings)

        batch_indices = torch.arange(
            input_ids.size(0),
            device=input_ids.device
        )

        masked_word_features = encoded_output[
            batch_indices,
            mask_positions
        ]

        predictions = self.output_layer(masked_word_features)

        return predictions


# ---------------------------------------------------------
# 6. Set up training
# ---------------------------------------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SimpleMaskedLanguageModel(
    vocabulary_size=VOCAB_SIZE
).to(device)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

NUMBER_OF_EPOCHS = 300


# ---------------------------------------------------------
# 7. Train the model
# ---------------------------------------------------------
model.train()

for epoch in range(NUMBER_OF_EPOCHS):
    total_loss = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        mask_positions = batch["mask_position"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()

        predictions = model(
            input_ids,
            mask_positions
        )

        loss = loss_function(
            predictions,
            targets
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 25 == 0:
        average_loss = total_loss / len(data_loader)

        print(
            f"Epoch {epoch + 1}/{NUMBER_OF_EPOCHS}, "
            f"Loss: {average_loss:.4f}"
        )


# ---------------------------------------------------------
# 8. Predict a masked word
# ---------------------------------------------------------
def predict_masked_word(sentence: str, top_k: int = 5):
    model.eval()

    words = sentence.lower().split()

    if "[mask]" not in words:
        raise ValueError(
            "The sentence must contain the token [MASK]."
        )

    mask_position = words.index("[mask]")

    token_ids = []

    for word in words:
        if word == "[mask]":
            token_ids.append(MASK_ID)
        else:
            token_ids.append(
                word_to_id.get(word, UNK_ID)
            )

    if len(token_ids) > MAX_LENGTH:
        raise ValueError(
            f"Sentence is too long. Maximum length is {MAX_LENGTH} words."
        )

    token_ids = pad_sequence(token_ids)

    input_tensor = torch.tensor(
        [token_ids],
        dtype=torch.long
    ).to(device)

    position_tensor = torch.tensor(
        [mask_position],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        predictions = model(
            input_tensor,
            position_tensor
        )

        probabilities = torch.softmax(
            predictions,
            dim=1
        )

        top_probabilities, top_indices = torch.topk(
            probabilities,
            k=min(top_k, VOCAB_SIZE),
            dim=1
        )

    print(f"\nInput: {sentence}")
    print("Predictions:")

    for probability, word_id in zip(
        top_probabilities[0],
        top_indices[0]
    ):
        predicted_word = id_to_word[word_id.item()]

        if predicted_word not in special_tokens:
            print(
                f"{predicted_word}: "
                f"{probability.item():.4f}"
            )


# ---------------------------------------------------------
# 9. Test the model
# ---------------------------------------------------------
predict_masked_word(
    "the cat is sitting on the [MASK]"
)

predict_masked_word(
    "the programmer is writing [MASK] code"
)

predict_masked_word(
    "the bird is flying in the [MASK]"
)


# ---------------------------------------------------------
# 10. Save the trained model
# ---------------------------------------------------------
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "word_to_id": word_to_id,
        "id_to_word": id_to_word,
        "max_length": MAX_LENGTH
    },
    "simple_mlm_model.pth"
)

print("\nModel saved as simple_mlm_model.pth")

In [ ]:
Training_data.txt


the cat is sitting on the mat
the dog is playing in the garden
the boy is reading a book
the girl is eating an apple
the teacher is writing on the board
the student is learning machine learning
the car is parked near the house
the bird is flying in the sky
the baby is sleeping in the room
the man is drinking a cup of tea
the woman is cooking food
the children are playing football
the computer is processing the data
the scientist is conducting an experiment
the programmer is writing python code